# STEP 1 — 데이터 확보

AI Hub 561번 「반려동물 피부 질환 데이터」에서 **반려견 + 일반카메라** 부분만 받습니다.

## 시작 전 체크리스트

1. [AI Hub](https://aihub.or.kr) 회원가입
2. [해당 데이터셋 페이지](https://aihub.or.kr/aihubdata/data/view.do?dataSetSn=561)에서 **활용신청** → 승인 대기 (보통 1영업일)
3. 마이페이지에서 **API Key 발급** (이메일로 옵니다)
4. 발급받은 키를 아래 위치에 등록

| 환경 | 등록 위치 |
|---|---|
| Colab | 왼쪽 사이드바 🔑 **보안 비밀** → 이름 `AIHUB_API_KEY` → **노트북 액세스** 토글 ON |
| Kaggle | Add-ons → **Secrets** → 이름 `AIHUB_API_KEY` → 이 노트북에 Attach |

> ⚠️ **키를 노트북 셀에 직접 붙여넣지 마세요.** 노트북을 공유하거나 GitHub 에 올리는 순간
> 키가 그대로 노출됩니다. Secrets 에 넣으면 코드에는 이름만 남습니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    BASE = "/content" if os.path.isdir("/content") else (
           "/kaggle/working" if os.path.isdir("/kaggle/working") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

## 1. aihubshell 설치

AI Hub 가 제공하는 공식 CLI 입니다. 웹에서 클릭으로 받는 것보다 훨씬 편하고,
무엇보다 **필요한 파일만 골라 받을 수 있습니다.**

In [ ]:
from src import aihub

aihub.install()
APIKEY = env.secret("AIHUB_API_KEY")   # ← 값은 화면에 찍히지 않습니다
print("API Key 로드 완료 (길이:", len(APIKEY), ")")

## 2. 파일 목록 확인

`-mode l` 로 데이터셋 안에 어떤 파일이 있는지 먼저 봅니다.

전체가 500GB 급이라 **절대 통째로 받으면 안 됩니다.** Colab 디스크는 100GB 남짓이고
세션이 끊기면 다 날아갑니다.

In [ ]:
files = aihub.list_files(APIKEY)

### 파싱이 0건이면?

AI Hub 출력 포맷이 바뀐 것입니다. 아래로 원본을 직접 보고 filekey 를 눈으로 고른 뒤,
다음 셀에서 `picks` 를 수동으로 지정하세요.

In [ ]:
# 파싱 실패했을 때만 실행
# print(aihub.raw_listing(APIKEY)[:6000])

## 3. 반려견 + 일반카메라만 선별

`max_gb` 로 한 번에 받을 용량을 제한합니다. 디스크가 차면 STEP 3(전처리)로 용량을 줄인 뒤
이 노트북을 다시 돌려 나머지를 받으면 됩니다 — 이미 받은 건 자동으로 건너뜁니다.

In [ ]:
picks = aihub.select_files(files, max_gb=40)

# 수동 지정 예시 (자동 선별이 안 될 때):
# picks = ["51937", "51938", "51939"]

## 4. 다운로드

`chunk=1` 이라 파일 하나씩 받고, 받을 때마다 기록을 남깁니다.
세션이 끊겨도 이 셀을 다시 실행하면 이어받습니다.

In [ ]:
failed = aihub.download(APIKEY, picks, chunk=1)

In [ ]:
aihub.unpack_all()      # 자동 해제 안 된 압축이 남아 있으면 마저 풀기
info = aihub.verify()   # 이미지/JSON 개수 확인

## 5. Google Drive 백업 (Colab, 선택)

Colab 세션은 끊기면 `/content` 가 통째로 사라집니다.
**원본 전체를 Drive 에 올리지는 마세요** — 무료 15GB 로는 어림도 없고, Drive I/O 가 느려서
학습이 오히려 느려집니다.

권장: 원본은 세션 디스크에 두고, **STEP 3에서 만든 크롭본 + 매니페스트만** Drive 에 백업.

In [ ]:
# DRIVE = env.mount_drive()
# print(DRIVE)

---
## ✅ 다음 단계

`01_데이터_스캔_EDA.ipynb` 로 넘어가세요.

거기서 **데이터가 실제로 어떻게 생겼는지** 알아냅니다.
전처리 코드를 짜기 전에 반드시 거쳐야 하는 단계입니다.

📖 함께 읽기: [`docs/cautions/01_데이터_라이선스와_재배포_금지.md`](../docs/cautions/01_데이터_라이선스와_재배포_금지.md)